<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">

# Governance and safe rollout

*Draft notebook — a narrative layer over the tested `src/`, not a re-implementation. The governance artifacts (`model_card`, `registry`, `lineage`) and the rollout are the chapter's own code. The champion/challenger split is evaluated by the **real AWS AppConfig agent** — the notebook starts that published agent image with docker-py (the same image the compose stack and ECS run), so the bucketing here is authoritative, not a reimplementation. On AWS the same agent runs as a Fargate sidecar — see `aws/`.*

<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">

<h2><i class="fas fa-cog" style="color:#18ab4b"></i>&nbsp; Parameters</h2>

Tagged `parameters` for papermill.

In [1]:
# parameters
MODE = "local"  # "local": start the agent with docker-py  |  "aws": use the deployed AppConfig
REGION = "us-east-1"
AGENT_IMAGE = "public.ecr.aws/aws-appconfig/aws-appconfig-agent:2.x"
N_APPLICATIONS = 300

In [2]:
import json
import sys
import pandas as pd

# the tested governance code, imported not copied
sys.path.insert(0, "../src")
from governance import model_card, registry, lineage

models = json.loads(open("../governance/models.json").read())
list(models)

['credit-scorecard', 'credit-challenger']

<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">

<h2><i class="fas fa-clipboard-check" style="color:#18ab4b"></i>&nbsp; The audit record: registry, model cards, lineage</h2>

The record an auditor reads: which models exist and their champion/challenger stage (the **registry**), what each model is and how it was evaluated (its **model card**), and what it was trained on and where it runs (its **lineage**). All rendered from the facts a team owns in `governance/models.json` — real chapter-2 metrics, nothing fabricated.

In [3]:
reg = registry(models)
pd.DataFrame(reg["model_package_groups"])[
    ["model_package_group_name", "stage", "latest_version", "metrics"]
]

,model_package_group_name,stage,latest_version,metrics
0,credit-scorecard,champion,1,"{'AUC': 0.8625, 'Gini': 0.725, 'KS': 0.5681}"
1,credit-challenger,challenger,1,"{'AUC': 0.883, 'Gini': 0.766, 'KS': 0.6041}"


The challenger's model card — overview, intended use and risk rating, and evaluation:

In [4]:
card = model_card("credit-challenger", models["credit-challenger"])
pd.Series(
    {
        "algorithm": card["model_overview"]["algorithm_type"],
        "problem": card["model_overview"]["problem_type"],
        "risk_rating": card["intended_uses"]["risk_rating"],
        "intended_use": card["intended_uses"]["purpose_of_model"],
    }
).to_frame("credit-challenger")

,credit-challenger
algorithm,Monotone CatBoost
problem,Binary classification (probability of default)
risk_rating,High
intended_use,Candidate replacement for the incumbent scorec...


In [5]:
metrics = card["evaluation_details"][0]["metric_groups"][0]["metric_data"]
pd.DataFrame(metrics)[["name", "value"]].set_index("name").T

name,AUC,Gini,KS
value,0.883,0.766,0.6041


Lineage: dataset to model to the endpoint that serves it.

In [6]:
pd.DataFrame(lineage("credit-challenger", models["credit-challenger"])["associations"])

,source,type,action
0,Synthetic credit-bureau applications (chapter ...,DataSet,ContributedTo
1,train-credit-challenger,TrainingJob,Produced
2,credit-challenger:1,Model,DeployedTo
3,credit-rollout-gateway,Endpoint,Serves


<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">

<h2><i class="fas fa-code-branch" style="color:#18ab4b"></i>&nbsp; Champion/challenger rollout, bucketed by the real agent</h2>

The `challenger_rollout` feature flag sends a share of applications to the challenger with a deterministic `split` rule, so the same loan always lands the same way and widening the rollout is a flag edit. The split is evaluated by the AWS AppConfig agent, not by Python; below, docker-py starts that agent from the committed Ion flag file and the notebook asks it for each loan's variant — exactly the call the gateway makes on ECS.

In [7]:
import os
import time
from collections import Counter
import docker

from appconfig import get_appconfig

configs = os.path.abspath("../local/agent-configs")
client = docker.from_env()
client.images.pull(AGENT_IMAGE)
agent = client.containers.run(
    AGENT_IMAGE,
    environment={"LOCAL_DEVELOPMENT_DIRECTORY": "/configs"},
    volumes={configs: {"bind": "/configs", "mode": "ro"}},
    ports={"2772/tcp": 2772},
    detach=True,
    auto_remove=True,
)
time.sleep(5)  # the agent reads the Ion file once at startup
f"agent {agent.short_id} serving on :2772"

'agent 1b891c8d1adc serving on :2772'

In [8]:
ac = get_appconfig()  # AgentAppConfig at localhost:2772
variants, models_used = Counter(), Counter()
try:
    for line in open("../data/applications.jsonl").read().splitlines()[:N_APPLICATIONS]:
        loan = json.loads(line)
        flag = ac.configuration({"loanId": str(loan["loanId"])})["challenger_rollout"]
        variants[flag["_variant"]] += 1
        models_used[flag.get("model", "credit-scorecard")] += 1
finally:
    agent.stop()

total = sum(variants.values())
pd.DataFrame(
    {
        "applications": dict(variants),
        "share": {k: f"{v / total:.0%}" for k, v in variants.items()},
    }
)

,applications,share
champion,238,79%
challenger,62,21%


The same loan ids fed to the agent on ECS produce the same buckets: the split is a property of the agent, so it is identical in the notebook, the compose stack, and the cloud.

<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">

<h2><i class="fas fa-cloud" style="color:#18ab4b"></i>&nbsp; On AWS</h2>

In `local` mode the cells above run the agent under docker-py and render the governance artifacts in-process. On AWS the same agent is a Fargate sidecar next to the gateway, the flag lives in AWS AppConfig (widened with `make -C aws flip PCT=...`, no redeploy), and `governance.py --upload` writes these cards, registry versions, and lineage to SageMaker.

---
*The rollout is bucketed by the real AWS AppConfig agent both here and on ECS, and the governance artifacts are the same records `governance.py` uploads to SageMaker. This notebook is the narrative view of the tested `src/`.*